In [1]:
# import
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import json
from datetime import datetime, timedelta
from dateutil import parser

In [2]:
# 설정
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

MAX_PAGES = 5
BASE_DIR = "inven_out"

# 실제 게시판 URL
LOSTARK_MOBILE_URL = "https://www.inven.co.kr/board/lostarkmobile/6389"
AION2_URL = "https://www.inven.co.kr/board/aion2/6388?my=chuchu"

# 아이온2 출시 기준 2주
AION2_RELEASE_DATE = datetime(2025, 11, 19)
AION2_START = AION2_RELEASE_DATE - timedelta(days=14)
AION2_END = AION2_RELEASE_DATE + timedelta(days=14)

def sleep():
    time.sleep(random.uniform(1.0, 2.0))

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def load_state(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"done_urls": [], "data": []}

def save_state(path, state):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

In [3]:
# 링크 수집
def get_post_links(base_url, page):
    url = f"{base_url}?p={page}"
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    links = []
    rows = soup.select("table tbody tr")
    for row in rows:
        a_tag = row.select_one("td.tit a")
        if not a_tag:
            continue
        title = a_tag.text.strip()
        link = a_tag.get("href")
        links.append((title, link))
    return links

# 상세 수집
def get_post_detail(title, url):
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    content_tag = soup.select_one(".contentBody")
    content = content_tag.text.strip() if content_tag else ""

    created_at = ""
    date_tag = soup.select_one(".articleDate")
    if date_tag:
        created_at = date_tag.text.strip()

    views = 0
    view_tag = soup.select_one(".articleHit")
    if view_tag:
        try:
            views = int(view_tag.text.replace(",", "").strip())
        except:
            pass

    likes = 0
    like_tag = soup.select_one("div.recommend span.num")
    if like_tag:
        try:
            likes = int(like_tag.text.strip())
        except:
            pass

    comments = []
    comment_tags = soup.select(".cmtContent")
    for c in comment_tags:
        text = c.text.strip()
        if text:
            comments.append(text)

    return {
        "title": title,
        "content": content,
        "comments": comments,
        "url": url,
        "created_at": created_at,
        "views": views,
        "likes": likes,
        "comment_count": len(comments)
    }

# 크롤링 (중간 저장 + 진행률 표시)
def crawl_inven(base_url, game_type, filter_dates=None):
    game_dir = os.path.join(BASE_DIR, game_type)
    ensure_dir(game_dir)

    state_path = os.path.join(game_dir, "state.json")
    state = load_state(state_path)

    done_urls = set(state["done_urls"])
    data = state["data"]

    all_links = []
    for page in range(1, MAX_PAGES + 1):
        print(f"[{game_type}] page {page}/{MAX_PAGES}")
        links = get_post_links(base_url, page)
        all_links.extend(links)
        sleep()

    total_posts = len(all_links)
    print(f"[{game_type}] total {total_posts} posts")

    for idx, (title, link) in enumerate(all_links, 1):
        if link in done_urls:
            continue

        try:
            post_data = get_post_detail(title, link)

            # 아이온2 날짜 필터
            if filter_dates:
                try:
                    post_dt = datetime.strptime(post_data['created_at'], "%Y-%m-%d %H:%M")
                    if post_dt < filter_dates[0] or post_dt > filter_dates[1]:
                        continue
                except:
                    pass

            # 진행률 표시
            percent = round(idx / total_posts * 100, 1)
            print(f"[posts] {idx}/{total_posts} ({percent}%)")
            print(f"[comments] {link.split('/')[-1]} -> {post_data['comment_count']} comments")

            data.append(post_data)
            done_urls.add(link)

            # 중간저장
            save_state(state_path, {"done_urls": list(done_urls), "data": data})
            sleep()
        except Exception as e:
            print("error:", e)

    return data

# CSV 저장
def save_to_csv(data, game_type):
    game_dir = os.path.join(BASE_DIR, game_type)
    ensure_dir(game_dir)

    today = datetime.now().strftime("%Y%m%d")
    filename = os.path.join(game_dir, f"{game_type}_{today}.csv")

    rows = []
    for post in data:
        if post["comments"]:
            for c in post["comments"]:
                rows.append({
                    "title": post["title"],
                    "content": post["content"],
                    "comment": c,
                    "url": post["url"],
                    "created_at": post["created_at"],
                    "views": post["views"],
                    "likes": post["likes"],
                    "comment_count": post["comment_count"]
                })
        else:
            rows.append({
                "title": post["title"],
                "content": post["content"],
                "comment": "",
                "url": post["url"],
                "created_at": post["created_at"],
                "views": post["views"],
                "likes": post["likes"],
                "comment_count": post["comment_count"]
            })

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved {filename}")

In [18]:
# 실행

# 로스트아크 모바일
lostark_data = crawl_inven(LOSTARK_MOBILE_URL, "lostark_mobile")
save_to_csv(lostark_data, "lostark_mobile")

# 아이온2
aion2_data = crawl_inven(
    AION2_URL,
    "aion2",
    filter_dates=(AION2_START, AION2_END)
)
save_to_csv(aion2_data, "aion2")

print("Done crawling")

[lostark_mobile] page 1/5
[lostark_mobile] page 2/5
[lostark_mobile] page 3/5
[lostark_mobile] page 4/5
[lostark_mobile] page 5/5
[lostark_mobile] total 226 posts
[posts] 1/226 (0.4%)
[comments] 1176 -> 0 comments
[posts] 2/226 (0.9%)
[comments] 1229 -> 0 comments
[posts] 3/226 (1.3%)
[comments] 1197 -> 0 comments
[posts] 4/226 (1.8%)
[comments] 1193 -> 0 comments
[posts] 5/226 (2.2%)
[comments] 1191 -> 0 comments
[posts] 6/226 (2.7%)
[comments] 1190 -> 0 comments
[posts] 7/226 (3.1%)
[comments] 1189 -> 0 comments
[posts] 8/226 (3.5%)
[comments] 1188 -> 0 comments
[posts] 9/226 (4.0%)
[comments] 1187 -> 0 comments
[posts] 10/226 (4.4%)
[comments] 1186 -> 0 comments
[posts] 11/226 (4.9%)
[comments] 1185 -> 0 comments
[posts] 12/226 (5.3%)
[comments] 1184 -> 0 comments
[posts] 13/226 (5.8%)
[comments] 1181 -> 0 comments
[posts] 14/226 (6.2%)
[comments] 1180 -> 0 comments
[posts] 15/226 (6.6%)
[comments] 1179 -> 0 comments
[posts] 16/226 (7.1%)
[comments] 1178 -> 0 comments
[posts] 17/226

In [4]:
import os
import json
import time
import random
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import requests
import pandas as pd
from bs4 import BeautifulSoup

In [67]:
"""===============================================
트러블 슈팅 : 아이온2 데이터 수집 오류 확인
원인 : 날짜 파싱 오류, 웹 크롤링 방식에서 URL 정상적으로 못 가져오는 현상

requests로 못 가져오고 브라우저 기반 수집(Selenium) 방법 선택
날짜 필터 유연화 (dateutil.parser)
중간 저장 (state.json) 적용
진행률 표시
CSV 저장
MAX_PAGES 확장 (2주치 게시글 확보)
==============================================="""
# Imports
import os
import time
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

# 기본 설정
BASE_URL = "https://www.inven.co.kr/board/aion2/6388"
OUTDIR = "inven_out/aion2"
STATE_POSTS = os.path.join(OUTDIR, "state_posts.json")
STATE_COMMENTS = os.path.join(OUTDIR, "state_comments.json")
MAX_PAGE = 100
SLEEP = 0.5

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

os.makedirs(OUTDIR, exist_ok=True)

# 상태 저장/로드
def save_state(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_state(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

# 게시글 수집 함수
def get_posts_on_page(page:int):
    params = {"page":page}
    r = requests.get(BASE_URL, headers=HEADERS, params=params)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    posts = []
    rows = soup.select("table.bbs_list tbody tr")
    for tr in rows:
        no_tag = tr.select_one("td.no")
        title_tag = tr.select_one("td.subject a")
        if not no_tag or not title_tag:
            continue
        posts.append({
            "no": no_tag.text.strip(),
            "title": title_tag.text.strip(),
            "url": "https://www.inven.co.kr" + title_tag["href"],
            "author": tr.select_one("td.writer").text.strip(),
            "date": tr.select_one("td.date").text.strip(),
            "read": tr.select_one("td.read").text.strip(),
            "recommend": tr.select_one("td.recommend").text.strip(),
        })
    return posts

# 댓글 수집 함수
def get_comments_for_post(post_url:str):
    r = requests.get(post_url, headers=HEADERS)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    comments=[]
    for c in soup.select("div.comment_box div.reply"):
        author = c.select_one("span.user_name")
        text = c.select_one("div.reply_cont")
        if author and text:
            comments.append({
                "post_url":post_url,
                "comment_author":author.get_text(strip=True),
                "comment_text":text.get_text(strip=True),
            })
    return comments

# ---------------------------------------------------
# 게시글 수집
posts_state = load_state(STATE_POSTS)
all_posts = posts_state.get("posts",[])
done_pages = posts_state.get("done_pages",[])

for page in range(1, MAX_PAGE+1):
    if page in done_pages:
        print(f"[posts] page {page} already done")
        continue

    print(f"[posts] page {page} collecting...")
    posts = get_posts_on_page(page)
    if not posts:
        print("[posts] no more posts -> stop")
        break

    all_posts.extend(posts)
    done_pages.append(page)
    save_state(STATE_POSTS, {"posts":all_posts,"done_pages":done_pages})
    time.sleep(SLEEP)

df_posts = pd.DataFrame(all_posts)

# ---------------------------------------------------
# 댓글 수집
comments_state = load_state(STATE_COMMENTS)
all_comments = comments_state.get("comments",[])

for idx,post in enumerate(df_posts.itertuples(),1):
    print(f"[comments] {idx}/{len(df_posts)} -> {post.url}")
    comments = get_comments_for_post(post.url)
    all_comments.extend(comments)
    save_state(STATE_COMMENTS, {"comments":all_comments})
    time.sleep(SLEEP)

df_comments=pd.DataFrame(all_comments)

# ---------------------------------------------------
today = datetime.now().strftime("%Y%m%d")
posts_csv = os.path.join(OUTDIR,f"aion2_{today}_posts.csv")
comments_csv = os.path.join(OUTDIR,f"aion2_{today}_comments.csv")

df_posts.to_csv(posts_csv,index=False,encoding="utf-8-sig")
df_comments.to_csv(comments_csv,index=False,encoding="utf-8-sig")

print("Saved posts:",posts_csv)
print("Saved comments:",comments_csv)

[posts] page 1 collecting...
[posts] no more posts -> stop
Saved posts: inven_out/aion2\aion2_20260326_posts.csv
Saved comments: inven_out/aion2\aion2_20260326_comments.csv
